In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import csr_matrix, coo_matrix, hstack, bmat, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
from scipy.spatial.distance import pdist, squareform
import geopandas as gpd
import pyreadr

# ================================================================
# Load data
# ================================================================
no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
coords = all_y.iloc[:, :2].rename(columns={0:"LON",1:"LAT"}).to_numpy()

S, TT = y.shape
period = 52

# ================================================================
# Spatial adjacency matrix (BYM CAR structure)
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
rotated = (xy @ R.T) / 1e6

Distances = squareform(pdist(rotated))
Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

D = csr_matrix(np.diag(np.array(Omg.sum(axis=1)).flatten()))
prec = D - Omg


# ================================================================
# p01 event: y_t = 0 → y_{t+1} = 1
# ================================================================
loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]
kappa = next_y - 0.5

t = time_idx + 1

# ================================================================
# Covariates: ONLY seasonal components (8 cols)
# ================================================================
covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t/period), np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period), np.sin(2*np.pi*t/period),
    t, t
])

K = covariates.shape[1]  # 8

# ================================================================
# Design matrix (spatial BYM version)
# ================================================================
rows, cols, vals = [], [], []

for i in tqdm(range(N), desc="Building design_loc"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_mat = coo_matrix((vals,(rows,cols)), shape=(N, K*S)).tocsr()
theta_dim = K*S

# ================================================================
# MCMC setup
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 2
b_tau = 25

save_idx = 0

# ================================================================
# MCMC
# ================================================================
for it in tqdm(range(total_iters), desc="MCMC p01 BYM noTempLatElev"):

    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    # build precision blocks CAR/IID alternating
    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)  # CAR
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))  # IID

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            row.append(block_list[i] if i == j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec
    factor = cholesky(pos_prec)

    mu = factor.solve_A(design_mat.T.dot(kappa))
    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    # tau updates
    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ (prec.dot(beta)) if j % 2 == 0 else beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:,save_idx] = curr_theta
        all_tau[:,save_idx] = curr_tau
        save_idx += 1
        if save_idx >= tot_save:
            break

np.savez_compressed(
    r"D:\77\Research\temp\snow\bym01.npz",
    all_theta=all_theta,
    all_tau=all_tau
)


MCMC p01 BYM noTempLatElev:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_66192\3544292562.py:145: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC p01 BYM noTempLatElev:  62%|██████▏   | 3697/6000 [2:54:32<1:48:43,  2.83s/it]


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import csr_matrix, coo_matrix, hstack, bmat, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
from scipy.spatial.distance import pdist, squareform
import geopandas as gpd
import pyreadr

# ================================================================
# Load data
# ================================================================
no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
coords = all_y.iloc[:, :2].rename(columns={0:"LON",1:"LAT"}).to_numpy()

S, TT = y.shape
period = 52

# ================================================================
# Spatial adjacency matrix (same as p01)
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
rotated = (xy @ R.T) / 1e6

Distances = squareform(pdist(rotated))
Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

D = csr_matrix(np.diag(np.array(Omg.sum(axis=1)).flatten()))
prec = D - Omg

# ================================================================
# p10 event: y_t = 1 → y_{t+1} = 0
# ================================================================
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]
event = 1 - next_y
kappa = event - 0.5

t = time_idx + 1

# ================================================================
# Covariates ONLY (8 seasonal ones)
# ================================================================
covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t/period), np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period), np.sin(2*np.pi*t/period),
    t, t
])

K = covariates.shape[1]

# ================================================================
# Build design matrix
# ================================================================
rows, cols, vals = [], [], []

for i in tqdm(range(N), desc="Building design p10"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_mat = coo_matrix((vals,(rows,cols)), shape=(N, K*S)).tocsr()
theta_dim = K*S

# ================================================================
# MCMC
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 2
b_tau = 25

save_idx = 0

for it in tqdm(range(total_iters), desc="MCMC p10 BYM noTempLatElev"):

    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            row.append(block_list[i] if i == j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec
    factor = cholesky(pos_prec)

    mu = factor.solve_A(design_mat.T.dot(kappa))
    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    # tau updates
    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ (prec.dot(beta)) if j % 2 == 0 else beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:,save_idx] = curr_theta
        all_tau[:,save_idx] = curr_tau
        save_idx += 1
        if save_idx >= tot_save:
            break

np.savez_compressed(
    r"D:\77\Research\temp\snow\bym10.npz",
    all_theta=all_theta,
    all_tau=all_tau
)


MCMC p10 BYM noTempLatElev:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_115944\115398297.py:142: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC p10 BYM noTempLatElev: 100%|█████████▉| 5995/6000 [2:33:39<00:07,  1.54s/it]  
